# V6.1 Clean YOLO Baselines — 2×T4, resumable

Chạy notebook này **sau khi V6.1 freeze đã PASS**.

Frozen dataset:

- final tag: `v6p1_5d0b1149891c`
- base: `56b7ea194ee5`
- FOTL overlay: `ee2d3c4497fd`
- train / val / test: `17655 / 2263 / 2012`
- 6 classes:
  0. insulator
  1. insulator_broken
  2. insulator_flashover
  3. bird_nest
  4. foreign_object
  5. broken_strand

Notebook này:
- reconstruct runtime union từ frozen base + frozen overlay;
- verify fingerprint/split/class distribution trước train;
- train YOLO11n@640 trước để có edge baseline nhanh;
- train YOLO11m@1024 với **cùng hyperparameters** như clean baseline V6 cũ để giữ comparability;
- 2×T4;
- periodic checkpoint lên Camber;
- VAL only;
- TEST hoàn toàn untouched.

> Nếu Kaggle session hết giờ, chạy lại notebook. Nó tự resume từ Camber `last.pt`.


In [ ]:
from pathlib import Path
import os, sys, json, csv, shutil, subprocess, threading, time, tarfile
from collections import Counter

W = Path("/kaggle/working")
TMP = Path("/tmp/v6p1_train")
BASE_EXTRACT = TMP/"base"
OVERLAY_EXTRACT = TMP/"overlay"
RUNTIME = TMP/"runtime"
RUNS = W/"runs_v6p1"

for p in [TMP, BASE_EXTRACT, OVERLAY_EXTRACT, RUNS]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_TAG = "v6p1_5d0b1149891c"
BASE_TAG = "56b7ea194ee5"
OVERLAY_TAG = "ee2d3c4497fd"

DATA_ROOT = "stash://bunpmc/projects/yolov11sdi/datasets"
FREEZE_ROOT = f"{DATA_ROOT}/v6p1/{DATASET_TAG}"

BASE_STASH = f"{DATA_ROOT}/curated_v6_rs1280_{BASE_TAG}.tar"
OVERLAY_STASH = f"{FREEZE_ROOT}/v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"
FREEZE_STASH = f"{FREEZE_ROOT}/v6p1.freeze.json"
DIST_STASH = f"{FREEZE_ROOT}/v6p1_class_distribution.csv"

CKPT_ROOT = f"stash://bunpmc/projects/yolov11sdi/checkpoints/{DATASET_TAG}"

EXPECTED_NAMES = [
    "insulator",
    "insulator_broken",
    "insulator_flashover",
    "bird_nest",
    "foreign_object",
    "broken_strand",
]

EXPECTED_SPLITS = {
    "train": 17655,
    "val": 2263,
    "test": 2012,
}

# Run edge baseline first because it is faster.
RUN_YOLO11N = True
RUN_YOLO11M = True

SEED = 42
WORKERS = 4
UPLOAD_EVERY_MIN = 15


In [ ]:
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "ultralytics", "pyyaml", "pandas"
])

import torch
import pandas as pd
import yaml

print("PyTorch:", torch.__version__)
print("GPUs:", torch.cuda.device_count(), [
    torch.cuda.get_device_name(i)
    for i in range(torch.cuda.device_count())
])
assert torch.cuda.device_count() >= 2, "Bật Kaggle GPU T4 x2."


## 1. Camber auth + exact frozen records


In [ ]:
from kaggle_secrets import UserSecretsClient

key = UserSecretsClient().get_secret("CAMBER_API_KEY")
if not key:
    raise RuntimeError("Thiếu CAMBER_API_KEY trong Kaggle Secrets.")

os.environ["CAMBER_API_KEY"] = key

if shutil.which("camber") is None and not (Path.home()/".camber/bin/camber").exists():
    subprocess.check_call(
        "curl -sL https://cli.cambercloud.com/install-v2.sh | bash",
        shell=True
    )

os.environ["PATH"] = (
    f"{Path.home()}/.camber/bin:"
    f"{Path.home()}/.local/bin:"
    + os.environ.get("PATH","")
)

CAMBER = shutil.which("camber") or str(Path.home()/".camber/bin/camber")

def stash_cp(src, dst, required=True):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(dst)],
        text=True,
        capture_output=True,
        env=os.environ.copy()
    )
    ok = r.returncode == 0 and dst.exists()
    if required and not ok:
        raise RuntimeError(
            f"Camber cp failed:\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}"
        )
    return ok

def stash_put(src, remote, required=True):
    src = Path(src)
    if not src.exists():
        if required:
            raise FileNotFoundError(src)
        return False
    r = subprocess.run(
        [CAMBER, "stash", "cp", str(src), str(remote)],
        text=True,
        capture_output=True,
        env=os.environ.copy()
    )
    ok = r.returncode == 0
    if required and not ok:
        raise RuntimeError(
            f"Camber upload failed:\n{r.stdout[-1500:]}\n{r.stderr[-1500:]}"
        )
    if ok:
        print("Uploaded:", remote)
    return ok

freeze_local = TMP/"v6p1.freeze.json"
dist_local = TMP/"v6p1_class_distribution.csv"

stash_cp(FREEZE_STASH, freeze_local)
stash_cp(DIST_STASH, dist_local)

freeze = json.loads(freeze_local.read_text())

assert freeze["final_tag"] == DATASET_TAG
assert freeze["freeze_ready"] is True
assert freeze["base"]["tag"] == BASE_TAG
assert freeze["overlay"]["tag"] == OVERLAY_TAG
assert freeze["final_split_counts"] == EXPECTED_SPLITS

print(json.dumps(freeze, indent=2)[:4000])


## 2. Restore base + overlay and reconstruct runtime union

Không download từ web nữa. Training chỉ dùng đúng artifacts đã freeze trên Camber.


In [ ]:
base_tar = TMP/f"curated_v6_rs1280_{BASE_TAG}.tar"
overlay_tar = TMP/f"v6p1_fotl_overlay_{OVERLAY_TAG}.tar.gz"

if not any(BASE_EXTRACT.rglob("data.yaml")):
    print("Downloading frozen base...")
    stash_cp(BASE_STASH, base_tar)
    subprocess.check_call(["tar","-xf",str(base_tar),"-C",str(BASE_EXTRACT)])
    base_tar.unlink(missing_ok=True)

if not any(OVERLAY_EXTRACT.rglob("data.yaml")):
    print("Downloading frozen overlay...")
    stash_cp(OVERLAY_STASH, overlay_tar)
    subprocess.check_call(["tar","-xzf",str(overlay_tar),"-C",str(OVERLAY_EXTRACT)])
    overlay_tar.unlink(missing_ok=True)

def normalize_names(names):
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x:int(x))]
    return list(names)

def find_dataset_root(root):
    for yp in root.rglob("data.yaml"):
        try:
            cfg=yaml.safe_load(yp.read_text())
            if normalize_names(cfg.get("names",[])) == EXPECTED_NAMES:
                return yp.parent
        except Exception:
            pass
    raise RuntimeError(f"No matching dataset root under {root}")

BASE = find_dataset_root(BASE_EXTRACT)
OVERLAY = find_dataset_root(OVERLAY_EXTRACT)

print("BASE:", BASE)
print("OVERLAY:", OVERLAY)


In [ ]:
IMG_EXTS={".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}

def images_under(root):
    return sorted(
        p for p in Path(root).rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    )

def split_counts(root):
    return {
        sp: len(images_under(Path(root)/"images"/sp))
        for sp in ["train","val","test"]
    }

base_counts = split_counts(BASE)
overlay_counts = split_counts(OVERLAY)

print("Base:", base_counts)
print("Overlay:", overlay_counts)

assert base_counts == freeze["base"]["split_counts"]
assert overlay_counts == freeze["overlay"]["split_counts"]


In [ ]:
# Runtime union via symlink, no image duplication.
if RUNTIME.exists():
    shutil.rmtree(RUNTIME)

for sp in ["train","val","test"]:
    for src in ["base","fotl"]:
        (RUNTIME/"images"/sp/src).mkdir(parents=True, exist_ok=True)
        (RUNTIME/"labels"/sp/src).mkdir(parents=True, exist_ok=True)

def link_dataset(src_root, split, source_name):
    iroot=src_root/"images"/split
    lroot=src_root/"labels"/split

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        lp=lroot/rel.with_suffix(".txt")

        dip=RUNTIME/"images"/split/source_name/rel
        dlp=RUNTIME/"labels"/split/source_name/rel.with_suffix(".txt")

        dip.parent.mkdir(parents=True, exist_ok=True)
        dlp.parent.mkdir(parents=True, exist_ok=True)

        os.symlink(ip, dip)

        if lp.exists():
            os.symlink(lp, dlp)
        else:
            dlp.write_text("")

for sp in ["train","val","test"]:
    link_dataset(BASE, sp, "base")
    link_dataset(OVERLAY, sp, "fotl")

runtime_yaml = RUNTIME/"data.yaml"
runtime_yaml.write_text(yaml.safe_dump({
    "path": str(RUNTIME),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": EXPECTED_NAMES,
}, sort_keys=False))

runtime_counts = split_counts(RUNTIME)
print("Runtime:", runtime_counts)
assert runtime_counts == EXPECTED_SPLITS


## 3. Final data gate before training

Notebook recounts class boxes and compares them against the frozen distribution CSV.

Nếu lệch một box cũng dừng.


In [ ]:
actual_boxes=Counter()
actual_images=Counter()

for sp in ["train","val","test"]:
    iroot=RUNTIME/"images"/sp
    lroot=RUNTIME/"labels"/sp

    for ip in images_under(iroot):
        rel=ip.relative_to(iroot)
        lp=lroot/rel.with_suffix(".txt")

        seen=set()
        if lp.exists():
            for line in lp.read_text(errors="ignore").splitlines():
                if not line.strip():
                    continue
                parts=line.split()
                cid=int(float(parts[0]))
                vals=list(map(float,parts[1:5]))

                assert 0 <= cid < len(EXPECTED_NAMES), (lp,cid)
                assert all(0 <= x <= 1 for x in vals), (lp,vals)

                actual_boxes[(sp,cid)] += 1
                seen.add(cid)

        for cid in seen:
            actual_images[(sp,cid)] += 1

frozen_dist=pd.read_csv(dist_local)

for _,r in frozen_dist.iterrows():
    cid=int(r["class_id"])
    sp=str(r["split"])
    assert actual_boxes[(sp,cid)] == int(r["boxes"]), (
        sp,cid,actual_boxes[(sp,cid)],int(r["boxes"])
    )
    assert actual_images[(sp,cid)] == int(r["images_with_class"]), (
        sp,cid,actual_images[(sp,cid)],int(r["images_with_class"])
    )

print("✅ Exact class distribution gate passed.")
display(frozen_dist)


## 4. Resumable trainer

Hyperparameters are kept the same as the previous clean 6-class baseline where applicable.

This matters because the dataset changed. We do not want to change dataset **and** training recipe at the same time and then pretend we know which one caused the difference.


In [ ]:
from ultralytics import YOLO

class Uploader(threading.Thread):
    def __init__(self, local, remote, every_min=15):
        super().__init__(daemon=True)
        self.local=Path(local)
        self.remote=remote
        self.every=every_min*60
        self.stop_ev=threading.Event()
        self.last_mtime=None

    def push(self):
        if self.local.exists():
            m=self.local.stat().st_mtime
            if m != self.last_mtime:
                tmp=self.local.with_suffix(".upload.pt")
                shutil.copy2(self.local,tmp)
                try:
                    stash_put(tmp,self.remote)
                    self.last_mtime=m
                finally:
                    tmp.unlink(missing_ok=True)

    def run(self):
        while not self.stop_ev.wait(self.every):
            self.push()

    def stop(self):
        self.stop_ev.set()
        self.push()

def train_resumable(name, base_model, imgsz, epochs, batch, lr0, patience):
    run=RUNS/name
    wdir=run/"weights"
    wdir.mkdir(parents=True,exist_ok=True)

    best_l=wdir/"best.pt"
    last_l=wdir/"last.pt"

    remote=f"{CKPT_ROOT}/{name}"
    done_l=TMP/f"{name}.done.json"

    # Completed run: fetch and reuse.
    if stash_cp(f"{remote}/done.json",done_l,required=False):
        stash_cp(f"{remote}/best.pt",best_l)
        print(f"[{name}] already complete -> reuse best.pt")
        return best_l, json.loads(done_l.read_text())

    # Resume if local or remote last.pt exists.
    resume = last_l.exists() or stash_cp(
        f"{remote}/last.pt", last_l, required=False
    )

    uploader=Uploader(
        last_l,
        f"{remote}/last.pt",
        UPLOAD_EVERY_MIN
    )
    uploader.start()

    t0=time.time()

    try:
        if resume:
            print(f"[{name}] RESUME")
            YOLO(str(last_l)).train(resume=True)
        else:
            print(f"[{name}] FRESH TRAIN")
            YOLO(base_model).train(
                data=str(runtime_yaml),
                project=str(RUNS),
                name=name,
                exist_ok=True,
                device=[0,1],
                seed=SEED,
                workers=WORKERS,
                deterministic=False,
                imgsz=imgsz,
                epochs=epochs,
                batch=batch,
                optimizer="AdamW",
                lr0=lr0,
                lrf=0.10,
                cos_lr=True,
                warmup_epochs=1.0,
                warmup_bias_lr=0.0,
                weight_decay=5e-4,
                patience=patience,
                amp=True,
                cache=False,
                plots=False,
                mosaic=0.25,
                close_mosaic=5,
                mixup=0.0,
                copy_paste=0.0,
                scale=0.15,
                translate=0.05,
                hsv_h=0.01,
                hsv_s=0.35,
                hsv_v=0.25,
                fliplr=0.5,
            )
    finally:
        uploader.stop()

    if not best_l.exists() and last_l.exists():
        shutil.copy2(last_l,best_l)

    assert best_l.exists(), f"{name}: best.pt missing"

    info={
        "name":name,
        "base_model":base_model,
        "dataset_tag":DATASET_TAG,
        "dataset_final_sha256":freeze["final_sha256"],
        "base_tag":BASE_TAG,
        "overlay_tag":OVERLAY_TAG,
        "dataset_classes":EXPECTED_NAMES,
        "split_counts":EXPECTED_SPLITS,
        "imgsz":imgsz,
        "epochs_requested":epochs,
        "batch":batch,
        "optimizer":"AdamW",
        "lr0":lr0,
        "seconds":round(time.time()-t0),
    }

    csvp=run/"results.csv"
    if csvp.exists():
        rows=list(csv.DictReader(open(csvp)))
        if rows:
            info["last_epoch"]={
                k.strip():v.strip()
                for k,v in rows[-1].items()
            }

    stash_put(best_l,f"{remote}/best.pt")
    if last_l.exists():
        stash_put(last_l,f"{remote}/last.pt")
    if csvp.exists():
        stash_put(csvp,f"{remote}/results.csv")

    done_l.write_text(json.dumps(info,indent=2))
    stash_put(done_l,f"{remote}/done.json")

    return best_l, info


## 5. YOLO11n@640 edge baseline

Run first because it is much faster and immediately tells us whether the frozen V6.1 pipeline is sane.


In [ ]:
YOLO11N_BEST=None

if RUN_YOLO11N:
    YOLO11N_BEST,info_n=train_resumable(
        name="yolo11n_clean_640",
        base_model="yolo11n.pt",
        imgsz=640,
        epochs=50,
        batch=64,
        lr0=1e-3,
        patience=10,
    )
    print(json.dumps(info_n,indent=2)[:5000])


## 6. YOLO11m@1024 cloud baseline

Same training recipe as the previous clean V6 YOLO11m baseline, so the main deliberate change is the frozen dataset.


In [ ]:
YOLO11M_BEST=None

if RUN_YOLO11M:
    YOLO11M_BEST,info_m=train_resumable(
        name="yolo11m_clean_1024",
        base_model="yolo11m.pt",
        imgsz=1024,
        epochs=40,
        batch=16,
        lr0=3e-4,
        patience=8,
    )
    print(json.dumps(info_m,indent=2)[:5000])


## 7. VAL only

TEST remains untouched for the final selected models.


In [ ]:
def evaluate(best, imgsz, batch):
    if best is None:
        return None

    m=YOLO(str(best)).val(
        data=str(runtime_yaml),
        split="val",
        imgsz=imgsz,
        batch=batch,
        device=0,
        plots=False,
        verbose=False,
    )

    result={
        "precision":float(m.box.mp),
        "recall":float(m.box.mr),
        "mAP50":float(m.box.map50),
        "mAP50_95":float(m.box.map),
    }

    if hasattr(m.box,"maps"):
        result["per_class_mAP50_95"]={
            EXPECTED_NAMES[i]:float(v)
            for i,v in enumerate(m.box.maps)
            if i < len(EXPECTED_NAMES)
        }

    return result

results={
    "dataset_tag":DATASET_TAG,
    "dataset_final_sha256":freeze["final_sha256"],
    "base_tag":BASE_TAG,
    "overlay_tag":OVERLAY_TAG,
    "classes":EXPECTED_NAMES,
    "split_counts":EXPECTED_SPLITS,
    "note":"VAL only; TEST untouched",
}

if YOLO11N_BEST is not None:
    results["YOLO11n_clean_640"]=evaluate(
        YOLO11N_BEST,640,32
    )

if YOLO11M_BEST is not None:
    results["YOLO11m_clean_1024"]=evaluate(
        YOLO11M_BEST,1024,16
    )

print(json.dumps(results,indent=2))

rp=W/f"{DATASET_TAG}_yolo_clean_val_results.json"
rp.write_text(json.dumps(results,indent=2))
stash_put(rp,f"{CKPT_ROOT}/val_results.json")


## 8. What comes next

After this notebook finishes successfully, save the VAL metrics.

Then the next experiment is:

- RT-DETR-R50 B0 @640
- same frozen dataset tag `v6p1_5d0b1149891c`
- same train/val/test split
- TEST untouched

Do not alter V6.1 after this point. New data means V6.2.
